# Analysis of LSST Photometric Calibration Parameters

This notebook analyses the photometric calibration parameters attached to the
light curves by `05_AddLSSTCalibParam.ipynb`, for all stable SIMBAD stars,
across all six LSST bands.

## Physical motivation

The published source flux (e.g. `psfFlux`, in instrumental counts) is
converted to a calibrated flux in nanoJansky through the Butler `PhotoCalib`.
The working hypothesis explored here is:

$$
F_{nJy} = C(x, y, b) \cdot F_{\mathrm{instrum}} \cdot 10^{-0.4 (Z_b - Z_0)}
$$

where:
- $C(x, y, b)$ is the **local** photometric calibration factor for band $b$
  at the source pixel position (`calib_local`), with CCD-average
  `calib_mean` $\pm$ `calib_err`;
- $Z_b$ is the photometric zero-point for band $b$ (`zeropoint`), which is
  expected to include the grey atmospheric extinction (airmass-dependent
  absorption that is uniform over the field);
- $Z_0 = 31.4$ is the reference AB zero-point.

## What this notebook shows

For **each of the 6 bands** (u, g, r, i, z, y) one figure is produced with
5 stacked subplots (n rows × 1 column), all sharing the same MJD x-axis range
`[MJD_MIN, MJD_MAX]` (the global min/max over the full dataset):

| Row | Content |
|---|---|
| 0 | Airmass vs. MJD (secondary x-axis on top: human-readable dates) |
| 1 | Zero-point $Z_b$ vs. airmass |
| 2 | $C(x,y,b)$ vs. MJD, with the CCD-mean calibration $\pm$ its error overlaid |
| 3 | $(C(x,y,b) - C_{\mathrm{mean}}) / C_{\mathrm{mean}}$ vs. MJD (expected to scatter around zero) |
| 4 | Sky position: Dec (left axis) and RA (right axis) vs. MJD |

## Input
- `data_AddCalib_05_out/all_stars_lightcurves_calib.csv`

## Output
- `figs_AnaCalib_06/calib_analysis_band_<b>.pdf` / `.png` (one figure per band)

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-30
- **Last update:** 2026-06-30


## 1. Imports

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from astropy.time import Time

## 2. Logging setup

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ─────────────────────────────────────────────────────────────
NB_TAG = "AnaCalib_06"

# ── Input: enriched light curves with calibration (output of notebook 05) ────
DIR_DATA_IN = "./data_AddCalib_05_out"
GLOBAL_CALIB_FILE = "all_stars_lightcurves_calib.csv"

# ── Output ────────────────────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Directory ready: %s", DIR_FIGS)

# ── Reference AB zero-point ───────────────────────────────────────────────────
Z0 = 31.4

# ── Band order and display colors (LSST community convention) ─────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]
BAND_COLORS = {
    "u": "#56b4e9",
    "g": "#008060",
    "r": "#ff4000",
    "i": "#850000",
    "z": "#6600cc",
    "y": "#000000",
}

log.info("Configuration done. Z0 = %.2f", Z0)

## 4. Load the enriched light-curve table

In [ ]:
data_path = os.path.join(DIR_DATA_IN, GLOBAL_CALIB_FILE)
log.info("Loading: %s", data_path)
df = pd.read_csv(data_path)
log.info("Shape: %s", df.shape)
df.head(3)

In [ ]:
# Verify required columns
REQUIRED_COLS = [
    "band",
    "expMidptMJD",
    "airmass",
    "calib_mean",
    "calib_err",
    "calib_local",
    "zeropoint",
    "src_ra",
    "src_dec",
]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing from input file: {missing}")
log.info("All required columns present: %s", REQUIRED_COLS)

# Keep only rows with valid calibration
n_before = len(df)
df = df.dropna(subset=["calib_mean", "calib_local", "zeropoint", "airmass"]).copy()
n_after = len(df)
log.info(
    "Rows with valid calibration: %d / %d (%.1f %%)",
    n_after,
    n_before,
    100.0 * n_after / n_before if n_before else 0,
)

## 5. Global MJD range (shared by all subplots in all figures)

In [ ]:
MJD_MIN = df["expMidptMJD"].min()
MJD_MAX = df["expMidptMJD"].max()

# small padding for readability
mjd_pad = 0.02 * (MJD_MAX - MJD_MIN)
MJD_PLOT_MIN = MJD_MIN - mjd_pad
MJD_PLOT_MAX = MJD_MAX + mjd_pad

log.info(
    "Global MJD range: [%.3f, %.3f]  (padded: [%.3f, %.3f])", MJD_MIN, MJD_MAX, MJD_PLOT_MIN, MJD_PLOT_MAX
)

date_min_str = Time(MJD_MIN, format="mjd").iso[:10]
date_max_str = Time(MJD_MAX, format="mjd").iso[:10]
log.info("Date range: %s -> %s", date_min_str, date_max_str)

## 6. Bands present in the data

In [ ]:
bands_available = [b for b in BAND_ORDER if b in df["band"].unique()]
extra_bands = sorted(set(df["band"].unique()) - set(BAND_ORDER))
if extra_bands:
    log.warning("Unexpected bands found (not in standard ugrizy list): %s", extra_bands)
    bands_available += extra_bands

log.info("Bands to plot (%d): %s", len(bands_available), bands_available)
df["band"].value_counts().reindex(bands_available)

## 7. Helper: MJD -> readable date converter for the secondary x-axis

In [ ]:
def mjd_to_datetime(mjd_values):
    """Convert an array of MJD floats to matplotlib-compatible datetime objects."""
    return Time(mjd_values, format="mjd").datetime

## 8. Core plotting function: 5-row figure for one band

In [ ]:
def plot_calib_analysis_for_band(
    df: pd.DataFrame,
    band: str,
    mjd_min: float,
    mjd_max: float,
    z0: float,
    color: str,
    dir_figs: str,
) -> None:
    """Produce the 5-row calibration-diagnostic figure for a single band.

    Rows (top to bottom):
        0. airmass vs MJD          (secondary top x-axis: readable dates)
        1. zero-point vs airmass
        2. calib_local vs MJD      (calib_mean ± calib_err overlaid)
        3. (calib_local - calib_mean) / calib_mean vs MJD
        4. src_dec (left) & src_ra (right) vs MJD
    """
    sub = df.loc[df["band"] == band].sort_values("expMidptMJD")

    if len(sub) == 0:
        log.warning("No data for band '%s' — skipping figure.", band)
        return

    mjd = sub["expMidptMJD"].to_numpy()
    airmass = sub["airmass"].to_numpy()
    zeropoint = sub["zeropoint"].to_numpy()
    calib_mean = sub["calib_mean"].to_numpy()
    calib_err = sub["calib_err"].to_numpy()
    calib_local = sub["calib_local"].to_numpy()
    src_ra = sub["src_ra"].to_numpy()
    src_dec = sub["src_dec"].to_numpy()

    # Relative deviation of the local calib from the CCD-mean calib
    rel_dev = (calib_local - calib_mean) / calib_mean

    fig, axes = plt.subplots(5, 1, figsize=(12, 18), sharex=True)
    fig.suptitle(
        f"Photometric calibration diagnostics — band '{band}'  " f"({len(sub)} points)", fontsize=14, y=0.995
    )

    # ── Row 0: airmass vs MJD, with readable-date secondary top x-axis ────────
    ax0 = axes[0]
    ax0.scatter(mjd, airmass, s=10, alpha=0.6, color=color)
    ax0.set_ylabel("airmass")
    ax0.set_title("Airmass vs. time")
    ax0.grid(True, alpha=0.3)

    # Secondary top x-axis: human-readable dates
    ax0_top = ax0.twiny()
    ax0_top.set_xlim(ax0.get_xlim())
    tick_mjds = np.linspace(mjd_min, mjd_max, 6)
    tick_dates = Time(tick_mjds, format="mjd").datetime
    ax0_top.set_xticks(tick_mjds)
    ax0_top.set_xticklabels([d.strftime("%Y-%m-%d") for d in tick_dates], rotation=30, ha="left")
    ax0_top.set_xlabel("Date (UTC)")

    # ── Row 1: zero-point vs airmass ───────────────────────────────────────────
    ax1 = axes[1]
    ax1.scatter(airmass, zeropoint, s=10, alpha=0.6, color=color)
    ax1.set_xlabel("airmass")
    ax1.set_ylabel(r"zero-point $Z_b$  [AB mag]")
    ax1.set_title("Zero-point vs. airmass (probes grey extinction)")
    ax1.grid(True, alpha=0.3)
    # NOTE: this subplot intentionally uses airmass as x-axis (not MJD),
    # so it does not share the common MJD range of the other rows.

    # ── Row 2: calib_local vs MJD, with calib_mean ± calib_err overlaid ───────
    ax2 = axes[2]
    ax2.scatter(mjd, calib_local, s=8, alpha=0.4, color=color, label=r"$C(x,y)$ local")
    ax2.errorbar(
        mjd,
        calib_mean,
        yerr=calib_err,
        fmt=".",
        ms=3,
        color="black",
        alpha=0.5,
        elinewidth=0.5,
        label=r"$C_{\mathrm{mean}} \pm \sigma$ (CCD average)",
    )
    ax2.set_ylabel(r"$C(x,y,b)$  [nJy/ADU]")
    ax2.set_title("Local vs. CCD-mean calibration factor")
    ax2.legend(loc="best", fontsize=8)
    ax2.grid(True, alpha=0.3)

    # ── Row 3: relative deviation (calib_local - calib_mean)/calib_mean ───────
    ax3 = axes[3]
    ax3.scatter(mjd, rel_dev, s=8, alpha=0.5, color=color)
    ax3.axhline(0.0, color="black", lw=1, ls="--")
    ax3.set_ylabel(r"$(C_{\mathrm{local}} - C_{\mathrm{mean}}) / C_{\mathrm{mean}}$")
    ax3.set_title("Relative spatial deviation of local calibration")
    ax3.grid(True, alpha=0.3)

    # ── Row 4: sky position — dec (left) and ra (right) vs MJD ────────────────
    ax4 = axes[4]
    l1 = ax4.scatter(mjd, src_dec, s=8, alpha=0.6, color="tab:blue", label="Dec")
    ax4.set_ylabel("Dec  [deg]", color="tab:blue")
    ax4.tick_params(axis="y", labelcolor="tab:blue")
    ax4.set_xlabel("MJD")
    ax4.set_title("Sky position vs. time")
    ax4.grid(True, alpha=0.3)

    ax4_right = ax4.twinx()
    l2 = ax4_right.scatter(mjd, src_ra, s=8, alpha=0.6, color="tab:orange", label="RA")
    ax4_right.set_ylabel("RA  [deg]", color="tab:orange")
    ax4_right.tick_params(axis="y", labelcolor="tab:orange")

    # ── Common MJD x-axis range for rows 0, 2, 3, 4 ────────────────────────────
    for ax in [ax0, ax2, ax3, ax4]:
        ax.set_xlim(mjd_min, mjd_max)

    plt.tight_layout(rect=[0, 0, 1, 0.98])

    # ── Save ────────────────────────────────────────────────────────────────────
    out_name = f"calib_analysis_band_{band}"
    pdf_path = os.path.join(dir_figs, out_name + ".pdf")
    png_path = os.path.join(dir_figs, out_name + ".png")
    fig.savefig(pdf_path, dpi=150, bbox_inches="tight")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    log.info("Figure saved: %s (.pdf/.png)", os.path.join(dir_figs, out_name))

    plt.show()

## 9. Generate one figure per band

In [ ]:
for band in bands_available:
    color = BAND_COLORS.get(band, "gray")
    log.info("Plotting band '%s' (%d points)...", band, (df["band"] == band).sum())
    plot_calib_analysis_for_band(
        df,
        band=band,
        mjd_min=MJD_PLOT_MIN,
        mjd_max=MJD_PLOT_MAX,
        z0=Z0,
        color=color,
        dir_figs=DIR_FIGS,
    )

## 10. Diagnostics: per-band relative spatial deviation statistics

Quick numerical summary of $(C_{\mathrm{local}} - C_{\mathrm{mean}})/C_{\mathrm{mean}}$
to confirm it is indeed centred near zero with a spread reflecting the
CCD-scale flat-field / illumination correction.

In [ ]:
df["rel_dev_calib"] = (df["calib_local"] - df["calib_mean"]) / df["calib_mean"]

summary = (
    df.groupby("band")["rel_dev_calib"]
    .agg(n="count", mean="mean", std="std", median="median")
    .reindex(bands_available)
)
print("Relative spatial deviation of calib_local vs calib_mean, per band:")
display(summary)

## 11. Diagnostics: zero-point vs airmass — linear fit per band

If $Z_b$ includes grey atmospheric extinction, it should decrease
approximately linearly with airmass. A simple linear fit (slope = grey
extinction coefficient in mag/airmass) is reported per band.

In [ ]:
fit_results = []

for band in bands_available:
    sub = df.loc[df["band"] == band]
    if len(sub) < 2:
        continue
    x = sub["airmass"].to_numpy()
    y = sub["zeropoint"].to_numpy()
    slope, intercept = np.polyfit(x, y, 1)
    fit_results.append(
        dict(
            band=band,
            slope_mag_per_airmass=slope,
            intercept_zp_at_airmass0=intercept,
            n_points=len(sub),
        )
    )

df_fits = pd.DataFrame(fit_results).set_index("band").reindex(bands_available)
print("Linear fit: zeropoint = intercept + slope * airmass")
display(df_fits)

## 12. Output directory listing

In [ ]:
print(f"\n=== Contents of {DIR_FIGS} ===")
for entry in sorted(os.listdir(DIR_FIGS)):
    full = os.path.join(DIR_FIGS, entry)
    size_kb = os.path.getsize(full) / 1024
    print(f"  [FILE] {entry}  ({size_kb:.1f} kB)")